# ☁️ Python Cloud Data Platforms — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> AWS data platform is a city. S3 is the warehouse district — cheap storage for everything. Glue Catalog is the city directory — tells you what's where. Athena is a freelance investigator — you give it a question and it reads from the warehouse, charges by the page. Redshift is a dedicated analytics firm — expensive to run but blazing fast if you pre-arrange files on their desks. EMR is a temporary construction crew — spin up, do the heavy work, spin down.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Are Cloud Data Platforms? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: S3 — Partitioning Strategy & Lifecycle](#5) |
| 6 | [Pattern 2: AWS Glue — Catalog, Crawlers, ETL Jobs](#6) |
| 7 | [Pattern 3: Amazon Redshift — Distribution & Sort Keys](#7) |
| 8 | [Pattern 4: Amazon Athena — Cost Optimization & Pushdown](#8) |
| 9 | [Pattern 5: EMR Spark — Cluster Sizing & Cost](#9) |
| 10 | [The Cloud Platforms Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Are Cloud Data Platforms? The Visual Model

---

```
AWS DATA PLATFORM ARCHITECTURE

  SOURCES                INGEST              STORE              PROCESS
  ───────                ──────              ─────              ───────
  Databases    ─────►  Glue ETL    ─────►  S3 Data Lake  ◄──► EMR (Spark)
  APIs         ─────►  Kinesis     ─────►  (Parquet)     ◄──► Glue Jobs
  Logs         ─────►  DMS         ─────►  S3 Raw        ◄──► Athena (SQL)
  Streams      ─────►  Firehose    ─────►               ◄──► Redshift

  SERVE
  ─────
  Athena      → ad-hoc SQL on S3 ($5/TB scanned)
  Redshift    → BI dashboards, fast aggregations (pre-loaded)
  SageMaker   → ML model serving

S3 PATH PARTITIONING (performance critical):
  Bad:   s3://bucket/data/events.parquet  ← one file, full scan always
  Good:  s3://bucket/events/year=2024/month=03/day=15/part-00001.parquet
  Effect: WHERE event_date='2024-03-15' → reads only day=15 partition

REDSHIFT ARCHITECTURE:
  Leader Node:  receives queries, builds plan, coordinates workers
  Compute Nodes: hold slices of data, execute in parallel
  Distribution: each row goes to one node (via dist key hash)
  Sort key:     data physically sorted → zone map pruning per block

COST MODEL (approximate 2024 prices):
  S3 storage:      $0.023/GB/month
  Athena queries:  $5.00/TB scanned  ← partition + columnar = 90% cost reduction
  Redshift:        $0.25/node-hour (dc2.large) — always running
  EMR (spot):      60–80% discount vs on-demand for batch workloads
  Glue:            $0.44/DPU-hour (DPU = 4vCPU + 16GB)
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import random
import math
import hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from collections import defaultdict
import time

random.seed(42)

# simulate AWS S3 file system
@dataclass
class S3File:
    key:        str
    size_bytes: int
    row_count:  int
    format:     str  # 'parquet', 'csv', 'json'
    created_at: str

class S3Bucket:
    def __init__(self, name):
        self.name  = name
        self.files: List[S3File] = []

    def put(self, key, size_bytes, row_count, format='parquet', created_at='2024-01-01'):
        self.files.append(S3File(key, size_bytes, row_count, format, created_at))

    def list_prefix(self, prefix):
        return [f for f in self.files if f.key.startswith(prefix)]

    def total_bytes(self):
        return sum(f.size_bytes for f in self.files)

# create a simulated data lake bucket
lake = S3Bucket('data-lake-prod')

# populate with partitioned parquet files
for year in [2022, 2023, 2024]:
    for month in range(1, 13):
        for day in range(1, 29, 7):  # every 7 days
            size = random.randint(50_000_000, 200_000_000)  # 50-200 MB
            rows = size // 500  # ~500 bytes per row compressed
            lake.put(
                f'events/year={year}/month={month:02d}/day={day:02d}/part-00001.parquet',
                size, rows, 'parquet', f'{year}-{month:02d}-{day:02d}'
            )

total_gb = lake.total_bytes() / 1e9
print(f"Simulated S3 lake: {len(lake.files)} files, {total_gb:.1f} GB total")
print(f"Sample keys:")
for f in lake.files[:3]:
    print(f"  {f.key}  ({f.size_bytes/1e6:.1f}MB, {f.row_count:,} rows)")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
AWS DATA PLATFORM OPERATIONS
────────────────────────────────────────────────────────────────────────────
SERVICE    OPERATION                  WHAT IT DOES
────────────────────────────────────────────────────────────────────────────
S3         PutObject(key, data)       write file to bucket
           GetObject(key)             read file
           ListObjectsV2(prefix)      list files under prefix
           lifecycle_rule(days)       auto-transition to Glacier after N days
Glue       create_table(schema)       register table in catalog
           start_crawler()            auto-discover schema from files
           start_job_run(job_name)    run Spark ETL job on Glue workers
Athena     start_query(sql)           submit SQL query against S3 files
           get_query_results(id)      fetch results
           workgroup(config)          set max data scanned per query
Redshift   COPY FROM S3 (creds)       bulk load from S3 → columnar storage
           UNLOAD TO S3               export query results back to S3
           ANALYZE COMPRESSION        recommend encoding per column
EMR        run_job_flow(config)       launch Spark/Hadoop cluster
           add_step(spark_submit)     submit Spark job to running cluster
           terminate_cluster(id)      shut down (avoid idle charges)
────────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Store large files without partitioning — Athena scans everything
❌  Use Athena for 1000s of queries/day — cost adds up vs Redshift
❌  Keep EMR cluster running when idle — spin down between jobs
❌  Use Redshift distkey=ALL on large tables — duplicates data to all nodes
❌  Write thousands of tiny files to S3 — small file problem
❌  Use CSV on S3 for Athena — Parquet is 10× cheaper per query
```


In [ ]:
# Core API demo: S3 partition scan simulation

# partition filter reduces scanned files dramatically
def athena_scan(bucket, predicate_prefix):
    all_files = bucket.files
    matching  = bucket.list_prefix(predicate_prefix)
    scanned_bytes = sum(f.size_bytes for f in matching)
    total_bytes   = bucket.total_bytes()
    cost = scanned_bytes / 1e12 * 5.0  # $5 per TB
    pct  = 100 * scanned_bytes / total_bytes
    print(f"  prefix: '{predicate_prefix}'")
    print(f"    files scanned: {len(matching):3d}/{len(all_files)}")
    print(f"    bytes scanned: {scanned_bytes/1e9:.2f} GB / {total_bytes/1e9:.2f} GB ({pct:.1f}%)")
    print(f"    Athena cost:   ${cost:.4f}")

print("=== Athena Partition Scan Cost ===")
print("Query: WHERE year=2024 AND month=03 AND day=15")
athena_scan(lake, 'events/year=2024/month=03/')

print()
print("Query: WHERE year=2024 (all months)")
athena_scan(lake, 'events/year=2024/')

print()
print("Query: no partition filter (full table scan)")
athena_scan(lake, 'events/')

print()
total_cost_unoptimized = lake.total_bytes() / 1e12 * 5.0
march_bytes = sum(f.size_bytes for f in lake.list_prefix('events/year=2024/month=03/'))
# parquet columnar: only 3 of 20 columns needed → read ~15%
parquet_savings = march_bytes * 0.15  # columnar + compression
print("=== Format + Partition savings ===")
print(f"  Full table scan (CSV, no partition): ${total_cost_unoptimized:.2f}")
print(f"  Partition pruned (one month):        ${march_bytes/1e12*5:.4f}")
print(f"  + Parquet column pruning (~85% skip): ${parquet_savings/1e12*5:.5f}")
print(f"  Total savings vs worst case: ~{total_cost_unoptimized/(parquet_savings/1e12*5):.0f}×")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                     AWS SERVICE
────────────────────────────────────────────────────────────────────────────
Store raw files cheaply                   S3 Standard
Rarely accessed archive (> 90 days)       S3 Glacier Instant Retrieval
Register schema for SQL queries           Glue Data Catalog
Auto-discover schema from files           Glue Crawler
Spark ETL without managing clusters       Glue Job (serverless)
Ad-hoc SQL on S3 (infrequent)             Athena (pay per scan)
High-frequency BI queries (1000s/day)     Redshift (pay per cluster-hour)
Heavy batch Spark (custom libraries)      EMR (managed cluster)
CDC / real-time ingestion to lake         Kinesis Firehose → S3
Database migration to cloud               AWS DMS
────────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: S3 — Partitioning Strategy & Lifecycle

---

```
PROBLEM:
  A 10TB event log grows 100GB/day. Queries are always by date range.
  Design S3 storage to minimize Athena scan cost and storage cost.

APPROACH:
  Partition by query predicate columns. Lifecycle rules age out old data.

PARTITIONING STRATEGY:
  High-level: partition by columns that appear in WHERE clauses
  Rule: query by day → partition by year/month/day
  Rule: query by region → add region= partition
  Don't over-partition: > 10k partitions → slow metadata operations

PARTITION COLUMN ORDER RULE:
  Most selective first: year > month > day (not day > month > year)
  Reason: year pruning eliminates years of data; day alone doesn't help

S3 LIFECYCLE POLICY:
  raw/       → Standard (hot)
  curated/   → Standard (hot)
  archive/   → Glacier Instant (cold, >90 days) → $0.004/GB vs $0.023/GB

SLOW MOTION: partition routing for write
  event: {user_id:123, action:'click', ts:'2024-03-15T10:30:00'}
  extract: year=2024, month=03, day=15
  write to: s3://bucket/events/year=2024/month=03/day=15/part-N.parquet
  Athena query: WHERE ts BETWEEN '2024-03-15' AND '2024-03-17'
    → list partitions matching year=2024/month=03/day=15 and day=17
    → skip all other days — zero bytes read from other partitions

KEY INSIGHT:
  Partitioning is free at write time. The cost savings at query time are 10-100×.
  Always partition by the most common query predicate columns.

TIME / SPACE:
  Partition pruning: O(matching_partitions) files scanned vs O(total)
  Lifecycle savings: 80% storage cost reduction after 90 days (Glacier)
```


In [ ]:
# Pattern 1: S3 partitioning strategy simulation

# Slow motion: choosing partition strategy for event logs
# step 1: identify query patterns — most queries filter by date
# step 2: choose partition columns: year/month/day
# step 3: write events to partitioned paths
# step 4: query with date predicate → partition pruning eliminates 95%+ files

class S3PartitionedTable:
    """
    Cloud Platforms Pattern 1 — S3 partitioning strategy.
    Approach: Route files to partition prefixes; query with predicate pruning.
    Time:  O(P) files scanned where P = matching partitions (not total)
    Space: O(N) total data, organized by partition prefix
    """
    def __init__(self, bucket_name, partition_cols):
        self.bucket = bucket_name
        self.partition_cols = partition_cols
        self.files: List[S3File] = []
        self._counter = 0

    def write_partition(self, partition_values, size_bytes, row_count):
        # build partition path from column=value pairs
        path_parts = '/'.join(f'{col}={val}' for col, val in zip(self.partition_cols, partition_values))
        self._counter += 1
        key = f'{path_parts}/part-{self._counter:05d}.parquet'
        self.files.append(S3File(key, size_bytes, row_count, 'parquet', '2024-01-01'))
        return key

    def query(self, partition_predicates):
        # predicates is list of (col, value) pairs to filter
        pred_dict = dict(zip(self.partition_cols[:len(partition_predicates)], partition_predicates))
        matching = []
        for f in self.files:
            match = True
            for col, val in pred_dict.items():
                expected = f'{col}={val}'
                if expected not in f.key:
                    match = False; break
            if match:
                matching.append(f)
        total_bytes    = sum(f.size_bytes for f in self.files)
        matching_bytes = sum(f.size_bytes for f in matching)
        pct = 100 * matching_bytes / max(total_bytes, 1)
        cost = matching_bytes / 1e12 * 5.0
        print(f"    predicate: {pred_dict}")
        print(f"    scanned: {len(matching)}/{len(self.files)} files | {matching_bytes/1e9:.2f}GB/{total_bytes/1e9:.2f}GB ({pct:.1f}%)")
        print(f"    Athena cost: ${cost:.4f}")

# build the table — 3 years × 12 months × 4 days
table = S3PartitionedTable('data-lake', ['year', 'month', 'day'])
for yr in [2022, 2023, 2024]:
    for mo in range(1, 13):
        for day in [1, 8, 15, 22]:
            size = random.randint(80_000_000, 150_000_000)
            table.write_partition([yr, f'{mo:02d}', f'{day:02d}'], size, size//400)

total_gb = sum(f.size_bytes for f in table.files) / 1e9
print(f"Table: {len(table.files)} files, {total_gb:.1f} GB total")

print()
print("=== Query Scenarios ===")
print("  Scenario 1: single day")
table.query([2024, '03', '15'])

print()
print("  Scenario 2: single month")
table.query([2024, '03'])

print()
print("  Scenario 3: single year")
table.query([2024])

print()
print("  Scenario 4: no partition filter (full scan)")
table.query([])

print()
# lifecycle cost model
print("=== S3 Lifecycle Cost Modeling ===")
total_bytes   = sum(f.size_bytes for f in table.files)
hot_bytes     = total_bytes * 0.30   # last 90 days
glacier_bytes = total_bytes * 0.70   # older
standard_cost = total_bytes / 1e9 * 0.023
optimized_cost = hot_bytes / 1e9 * 0.023 + glacier_bytes / 1e9 * 0.004
print(f"  Total data: {total_bytes/1e9:.1f} GB")
print(f"  All-Standard cost:  ${standard_cost:.2f}/month")
print(f"  With Glacier (70%): ${optimized_cost:.2f}/month  (saves {(1-optimized_cost/standard_cost)*100:.0f}%)")

print("\nS3 partitioning pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: AWS Glue — Catalog, Crawlers, ETL Jobs

---

```
PROBLEM:
  You have 50 S3 paths of Parquet files with varying schemas.
  You need Athena to query them via SQL without manually defining schemas.

APPROACH:
  Glue Catalog = Hive Metastore compatible schema registry.
  Glue Crawler = auto-discovers schema from files → registers in Catalog.
  Glue ETL Job = serverless Spark for transformations.

GLUE CATALOG HIERARCHY:
  Catalog
  └── Database (namespace)
      └── Table
          ├── Schema (columns + types)
          ├── Location (s3://bucket/prefix/)
          ├── InputFormat (parquet/csv/json)
          └── Partition spec (columns that define partition paths)

CRAWLER WORKFLOW:
  1. Point crawler at s3://bucket/events/
  2. Crawler reads sample files → infers schema
  3. Crawler detects partitions: year=XXXX/month=XX/ → partition columns
  4. Registers table in Catalog: events (columns...) partitioned by (year, month, day)
  5. Athena can now query: SELECT * FROM database.events WHERE year=2024

GLUE JOB vs LAMBDA vs EMR:
  Glue Job:  serverless Spark, auto-scales, simple scripts → best for ETL
  Lambda:    15 min max, 10GB mem → only for tiny transformations
  EMR:       full control, custom libraries, spot instances → heavy batch

KEY INSIGHT:
  Glue Catalog is the glue between storage (S3) and query (Athena/Redshift Spectrum).
  Without it, Athena can't find tables. With it, schema is centrally managed.

TIME / SPACE:
  Crawler cost:   $0.44/DPU-hour, minimum 10 minutes
  ETL Job:        $0.44/DPU-hour, scales with data volume
  Catalog storage: free for first 1M objects
```


In [ ]:
# Pattern 2: Glue Catalog and Crawler simulation

# Slow motion: crawler discovers schema from S3 files
# step 1: crawler samples first file per partition → infers column types
# step 2: crawler detects partition columns from path: year=XXXX/month=XX
# step 3: merges schemas across files (adds new columns as nullable)
# step 4: registers table in catalog with full schema

@dataclass
class GlueCatalogTable:
    database:        str
    name:            str
    location:        str
    columns:         List[Dict]
    partition_cols:  List[str]
    input_format:    str
    num_files:       int = 0
    num_rows:        int = 0

class GlueCatalog:
    def __init__(self):
        self.tables: Dict[str, GlueCatalogTable] = {}

    def register_table(self, table: GlueCatalogTable):
        key = f"{table.database}.{table.name}"
        self.tables[key] = table

    def get_table(self, db, name) -> Optional[GlueCatalogTable]:
        return self.tables.get(f"{db}.{name}")

class GlueCrawler:
    def __init__(self, catalog: GlueCatalog, name: str, s3_target: str):
        self.catalog = catalog
        self.name = name
        self.s3_target = s3_target

    def _infer_type(self, value):
        # simple type inference from sample value
        if isinstance(value, int):   return 'int'
        if isinstance(value, float): return 'double'
        # check if string looks like date
        if isinstance(value, str) and len(value) == 10 and value[4] == '-': return 'date'
        return 'string'

    def crawl(self, sample_files: List[Dict], db_name: str, table_name: str):
        # step 1: infer schema from first file
        if not sample_files:
            return None
        sample_row = sample_files[0]
        columns = [{'name': k, 'type': self._infer_type(v)} for k, v in sample_row.items()
                   if not any(k.startswith(p) for p in ['year=', 'month=', 'day='])]

        # step 2: detect partition columns from S3 path structure
        partition_cols = ['year', 'month', 'day']  # inferred from path

        # step 3: count files and rows
        matching_files = table.files  # use our simulated lake
        num_rows = sum(f.row_count for f in matching_files)

        # step 4: register in catalog
        cat_table = GlueCatalogTable(
            database=db_name,
            name=table_name,
            location=self.s3_target,
            columns=columns,
            partition_cols=partition_cols,
            input_format='parquet',
            num_files=len(matching_files),
            num_rows=num_rows
        )
        self.catalog.register_table(cat_table)
        return cat_table

# simulate a Glue ETL job
class GlueETLJob:
    def __init__(self, name, dpu_count=10):
        self.name = name
        self.dpu_count = dpu_count  # each DPU = 4vCPU + 16GB RAM

    def run(self, input_count, transforms, output_count):
        duration_min = max(10, input_count / 1_000_000 * 2)  # ~2 min per 1M rows
        cost = self.dpu_count * (duration_min / 60) * 0.44
        print(f"  Job '{self.name}':")
        print(f"    DPUs: {self.dpu_count}, duration: {duration_min:.0f} min")
        print(f"    transforms: {transforms}")
        print(f"    input rows: {input_count:,} → output rows: {output_count:,}")
        print(f"    cost: ${cost:.2f}")

catalog  = GlueCatalog()
crawler  = GlueCrawler(catalog, 'events-crawler', f's3://{lake.name}/events/')

# sample rows to simulate crawler reading files
sample_rows = [{
    'event_id': i, 'user_id': random.randint(1,10000),
    'action': 'click', 'amount': random.uniform(1,500),
    'event_date': f'2024-03-{random.randint(1,28):02d}'
} for i in range(5)]

print("=== Glue Crawler Run ===")
discovered = crawler.crawl(sample_rows, 'analytics', 'events')
print(f"  Discovered table: {discovered.database}.{discovered.name}")
print(f"  Location: {discovered.location}")
print(f"  Columns: {[c['name'] + ':' + c['type'] for c in discovered.columns]}")
print(f"  Partitions: {discovered.partition_cols}")
print(f"  Stats: {discovered.num_files} files, {discovered.num_rows:,} rows")

print()
print("=== Glue ETL Job Simulation ===")
job = GlueETLJob('raw-to-curated', dpu_count=10)
job.run(
    input_count=50_000_000,
    transforms=['filter nulls', 'cast types', 'deduplicate', 'repartition by region'],
    output_count=48_500_000
)

print()
print("=== Glue vs Lambda vs EMR cost comparison ===")
rows = 50_000_000
print(f"  Processing {rows/1e6:.0f}M rows:")
print(f"  Glue (10 DPU, ~100min): ${10 * (100/60) * 0.44:.2f}")
print(f"  Lambda: impossible — 15 min timeout, 10GB limit")
print(f"  EMR (on-demand, 10 m5.xlarge, 1hr): ${10 * 1 * 0.192:.2f}")
print(f"  EMR (spot 70% discount): ${10 * 1 * 0.192 * 0.30:.2f}")

print("\nGlue catalog pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Amazon Redshift — Distribution & Sort Keys

---

```
PROBLEM:
  A Redshift query joining orders (10B rows) with customers (1M rows)
  takes 5 minutes. Most of the time is spent shuffling data between nodes.
  How do you eliminate the shuffle?

APPROACH:
  Distribution key (DISTKEY): determines which node stores each row.
  Sort key (SORTKEY): physical sort order within each node's slice.

DISTRIBUTION STYLES:
  EVEN:         round-robin — even spread, but always shuffles on joins
  KEY(col):     rows with same col value → same node — co-locates join data
  ALL:          copy full table to every node — good for small dim tables
  AUTO:         Redshift chooses (default — good starting point)

CO-LOCATION JOIN (zero shuffle):
  orders DISTKEY(customer_id) + customers DISTKEY(customer_id)
  → rows with customer_id=42 are on the SAME node for both tables
  → join is local — no network transfer between nodes

SORT KEY:
  Compound SORTKEY(date_col, region):  good for date range queries
  Interleaved SORTKEY(date, region):   equal weight on both columns
  Rule: use compound sort key matching most common WHERE predicates

ZONE MAP PRUNING:
  Each 1MB block stores min/max of sort key columns
  WHERE order_date > '2024-01-01': blocks with max_date < 2024-01-01 skipped
  Up to 95% of blocks skipped for highly selective date filters

KEY INSIGHT:
  Wrong distkey = data shuffled across nodes on every join.
  Right distkey (same as join key) = zero network shuffle.
  This is the single biggest Redshift performance lever.

TIME / SPACE:
  With shuffle:    O(N × M / nodes) + O(N) network transfer
  Co-located join: O(N × M / nodes²) — no network transfer
  Sort key block skip: O(blocks × selectivity) vs O(blocks)
```


In [ ]:
# Pattern 3: Redshift distribution and sort key simulation

# Slow motion: join with wrong distkey vs right distkey
# wrong: orders EVEN, customers EVEN → join needs all-to-all shuffle
# right: orders DISTKEY(cust_id), customers DISTKEY(cust_id) → join is local

class RedshiftCluster:
    """
    Cloud Platforms Pattern 3 — Redshift distribution and sort key simulation.
    Approach: Show data placement and shuffle cost with different dist strategies.
    Time:  O(N/nodes) per join with co-located distkey
           O(N×M/nodes + N_network) per join with wrong distkey
    Space: O(N) data across nodes (KEY), O(N×nodes) data with ALL style
    """
    def __init__(self, num_nodes):
        self.num_nodes = num_nodes

    def distribute(self, rows, strategy, key_col=None):
        nodes = defaultdict(list)
        for row in rows:
            if strategy == 'EVEN':
                node = hash(str(row)) % self.num_nodes  # round-robin-ish
            elif strategy == 'KEY':
                node = hash(str(row[key_col])) % self.num_nodes  # same key → same node
            elif strategy == 'ALL':
                for n in range(self.num_nodes):  # full copy to every node
                    nodes[n].append(row)
                continue
            nodes[node].append(row)
        return nodes

    def join_cost(self, left_nodes, right_nodes, join_col):
        # cost = rows that must be MOVED to co-locate with their join partner
        local_joins  = 0
        network_rows = 0
        for node_id in range(self.num_nodes):
            left_keys  = {r[join_col] for r in left_nodes.get(node_id, [])}
            right_keys = {r[join_col] for r in right_nodes.get(node_id, [])}
            # rows whose join partner is on a different node need network transfer
            matching_here = left_keys & right_keys
            local_joins  += len(matching_here)
            # right rows not on same node as their left partner must be shuffled
            for r in right_nodes.get(node_id, []):
                if r[join_col] not in left_keys:
                    network_rows += 1
        return local_joins, network_rows

cluster = RedshiftCluster(num_nodes=4)

# create test data
n_orders    = 1000
n_customers = 100
orders    = [{'order_id': i, 'customer_id': random.randint(1, n_customers), 'amount': i*10} for i in range(n_orders)]
customers = [{'customer_id': i, 'name': f'C{i}', 'tier': 'GOLD'} for i in range(1, n_customers+1)]

print("=== Distribution Strategy Comparison ===")
print(f"  {n_orders} orders, {n_customers} customers, {cluster.num_nodes} nodes")
print()

# EVEN distribution → shuffles on join
o_even = cluster.distribute(orders,    'EVEN')
c_even = cluster.distribute(customers, 'EVEN')
local_e, net_e = cluster.join_cost(o_even, c_even, 'customer_id')
print(f"  EVEN/EVEN distkey:")
print(f"    local joins: {local_e}  network shuffled rows: {net_e}  (SLOW — shuffle)")

# KEY distribution on same column → co-located joins
o_key = cluster.distribute(orders,    'KEY', 'customer_id')
c_key = cluster.distribute(customers, 'KEY', 'customer_id')
local_k, net_k = cluster.join_cost(o_key, c_key, 'customer_id')
print(f"  KEY(customer_id)/KEY(customer_id):")
print(f"    local joins: {local_k}  network shuffled rows: {net_k}  (FAST — co-located)")

# ALL distribution for small dimension table
o_key2 = cluster.distribute(orders,    'KEY', 'customer_id')
c_all  = cluster.distribute(customers, 'ALL')  # broadcast to all nodes
local_a, net_a = cluster.join_cost(o_key2, c_all, 'customer_id')
storage_overhead = n_customers * cluster.num_nodes
print(f"  KEY(orders)/ALL(customers):")
print(f"    local joins: {local_a}  network shuffled rows: {net_a}")
print(f"    storage overhead: {storage_overhead} rows ({cluster.num_nodes}× copies of customers)")

print()
print("=== Sort Key Zone Map Pruning ===")
# simulate blocks with sort key values
block_size = 100  # rows per 1MB block
sorted_orders = sorted(orders, key=lambda r: r['order_id'])
blocks = [sorted_orders[i:i+block_size] for i in range(0, len(sorted_orders), block_size)]
query_threshold = 500  # WHERE order_id > 500
skipped = sum(1 for b in blocks if b[-1]['order_id'] <= query_threshold)  # max < threshold
print(f"  {len(blocks)} blocks, WHERE order_id > {query_threshold}")
print(f"  Blocks skipped by zone map: {skipped}/{len(blocks)} ({100*skipped/len(blocks):.0f}%)")
print(f"  Rows avoided: {skipped * block_size} of {len(orders)}")

print("\nRedshift distkey pattern complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Amazon Athena — Cost Optimization & Pushdown

---

```
PROBLEM:
  An Athena query costs $50/run scanning 10TB of CSV data.
  How do you reduce it to < $0.05 without moving the data?

APPROACH:
  Two levers: partition pruning + file format. Together: ~1000× cost reduction.

COST REDUCTION TECHNIQUES (in order of impact):
  1. Partition — 90%+ reduction if predicate matches partition columns
  2. Columnar format (Parquet/ORC) — 80-90% reduction (column pruning)
  3. Compression (Snappy/ZSTD) — 3-5× reduction on top of columnar
  4. Athena workgroup — cap per-query scan to protect budget

SLOW MOTION: cost journey
  10TB CSV, no partition, no compression:
    Athena scans: 10TB × $5 = $50 per query

  10TB CSV, partitioned by date (query = 1 day = 1/365 of data):
    Athena scans: 10TB/365 = 27GB × $5 = $0.14 per query

  27GB Parquet (from 274GB daily CSV, 10× compressed):
    Athena scans: 2.7GB × $5 = $0.014 per query

  2.7GB Parquet, query needs 3/20 columns (15%):
    Athena scans: 0.41GB × $5 = $0.002 per query

  Total reduction: $50 → $0.002 = 25,000× cheaper

KEY INSIGHT:
  Every non-partitioned, non-columnar byte Athena reads costs money.
  Converting to Parquet + partitioning is the highest ROI optimization in AWS.

TIME / SPACE:
  Athena cost formula: $5 × (bytes_scanned / 1e12)
  Min charge: 10MB per query
  Parallelism: Athena scales automatically — more data = more workers, same latency
```


In [ ]:
# Pattern 4: Athena cost optimization simulation

# Slow motion: cost journey — each optimization layer reduces cost
# step 1: baseline (CSV, no partition): scan 100% of all data
# step 2: add partitioning: scan only matching partition files
# step 3: convert to Parquet: 10× compression → 10× fewer bytes
# step 4: column pruning: query needs 3/20 cols → read 15% of Parquet bytes

def athena_cost(bytes_scanned, label):
    min_bytes = 10 * 1024 * 1024  # 10MB minimum charge
    charged   = max(bytes_scanned, min_bytes)
    cost = charged / 1e12 * 5.0
    print(f"  {label:45s}: {bytes_scanned/1e9:8.2f} GB  ${cost:.4f}")
    return cost

print("=== Athena Cost Optimization Journey ===")
print(f"  Scenario: 10TB total data, query = 1 day, need 3 of 20 columns")
print()

total_tb       = 10
total_bytes    = total_tb * 1e12
days_in_data   = 365
day_fraction   = 1.0 / days_in_data
col_fraction   = 3.0 / 20.0
parquet_compression = 10.0  # 10× vs CSV

# baseline: CSV, no partition
c1 = athena_cost(total_bytes, "1. CSV, no partition (full scan)")

# add partition
partitioned_bytes = total_bytes * day_fraction
c2 = athena_cost(partitioned_bytes, "2. CSV, partitioned by day")

# convert to Parquet
parquet_bytes = partitioned_bytes / parquet_compression
c3 = athena_cost(parquet_bytes, "3. Parquet, partitioned by day")

# column pruning (15% of Parquet bytes for 3/20 columns)
pruned_bytes = parquet_bytes * col_fraction
c4 = athena_cost(pruned_bytes, "4. Parquet, partitioned, 3/20 cols")

print(f"\n  Total reduction: {c1/c4:,.0f}× cheaper")

print()
print("=== Workgroup Budget Controls ===")
workgroup_config = {
    'name': 'analytics-prod',
    'bytes_scanned_cutoff': 10 * 1024**3,  # 10GB max per query
    'enforce_workgroup_config': True,
    'publish_cloudwatch_metrics': True
}
print(f"  Workgroup '{workgroup_config['name']}':")
print(f"    Max scan per query: {workgroup_config['bytes_scanned_cutoff']/1e9:.0f} GB")
print(f"    Cost cap per query: ${workgroup_config['bytes_scanned_cutoff']/1e12*5:.2f}")
print(f"    Effect: query attempting to scan >10GB is cancelled automatically")

print()
print("=== Athena vs Redshift Decision ===")
queries_per_month = 1000
avg_scan_gb       = 0.1  # after optimization
athena_monthly    = queries_per_month * avg_scan_gb / 1000 * 5.0
redshift_monthly  = 2 * 24 * 30 * 0.25  # 2× dc2.large nodes, 24h/day, 30 days
print(f"  {queries_per_month} queries/month, avg {avg_scan_gb}GB scan each:")
print(f"  Athena monthly cost:    ${athena_monthly:.2f}")
print(f"  Redshift monthly cost:  ${redshift_monthly:.2f}  (2× dc2.large)")
print(f"  Break-even: ~{int(redshift_monthly/(avg_scan_gb/1000*5)):,} queries/month")
print(f"  Rule: < break-even → Athena. More frequent → Redshift.")

print("\nAthena cost optimization pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: EMR Spark — Cluster Sizing & Cost

---

```
PROBLEM:
  A nightly Spark job processes 500GB of data in 4 hours.
  Target: finish in 30 minutes. How do you size the cluster?

APPROACH:
  EMR cluster sizing = throughput requirement ÷ per-node throughput.
  Spot instances for 60-80% cost reduction on non-critical batch jobs.

CLUSTER COMPONENTS:
  Master node:  1× — orchestrates, doesn't process data (m5.xlarge)
  Core nodes:   N× — HDFS + Spark executors (r5.2xlarge for data-heavy)
  Task nodes:   M× — Spark only (no HDFS) — great for spot instances

SPARK EXECUTOR SIZING:
  r5.2xlarge: 8 vCPU, 64GB RAM
  Executors per node: 2 (leave 20% for OS + overhead)
    executor_cores: 3  (leaving 2 for overhead)
    executor_memory: 24GB  (leaving 16GB for OS)
  spark.default.parallelism: 2 × total_cores

SIZING CALCULATION:
  Current: 1 node, 4 hours, 500GB
  Target: 30 min = 8× speedup needed
  → Need 8× compute → 8 r5.2xlarge core nodes
  → Executors: 8 nodes × 2 executors = 16 executors
  → Partitions: 500GB / 128MB = 3900 → set spark.sql.shuffle.partitions=3900

SPOT INSTANCE STRATEGY:
  Master: on-demand (must not be interrupted)
  Core:   on-demand (HDFS must persist — use S3 as HDFS to enable spot cores)
  Task:   spot (stateless — can be interrupted, Spark re-runs lost tasks)

KEY INSIGHT:
  Read/write from S3 (not HDFS) → all nodes can be task nodes → all spot → 70% savings.

TIME / SPACE:
  Spark parallelism: O(N/partitions) per task
  Cluster cost: num_nodes × instance_cost × cluster_hours
  Spot savings: 60-80% vs on-demand
```


In [ ]:
# Pattern 5: EMR Spark cluster sizing and cost model

# Slow motion: calculate optimal cluster for a given job
# step 1: measure baseline (1 node, actual duration)
# step 2: compute target parallelism (data_size / partition_size)
# step 3: compute executor count needed to hit time target
# step 4: size cluster accordingly, assign spot where safe

@dataclass
class InstanceType:
    name:         str
    vcpu:         int
    memory_gb:    int
    on_demand_hr: float
    spot_hr:      float  # approximate spot price

INSTANCE_TYPES = {
    'm5.xlarge':  InstanceType('m5.xlarge',  4, 16,   0.192, 0.058),
    'r5.2xlarge': InstanceType('r5.2xlarge', 8, 64,   0.504, 0.151),
    'r5.4xlarge': InstanceType('r5.4xlarge', 16, 128, 1.008, 0.302),
    'm5.4xlarge': InstanceType('m5.4xlarge', 16, 64,  0.768, 0.230),
}

def size_emr_cluster(data_gb, target_minutes, partition_mb=128):
    print(f"  Job: {data_gb}GB data, target {target_minutes} min")
    # step 1: compute required partitions
    partitions = math.ceil(data_gb * 1024 / partition_mb)
    print(f"  Partitions: {partitions} ({partition_mb}MB each)")

    # step 2: estimate single-executor throughput
    # rough: 1 executor processes ~1 partition/minute for complex transforms
    executor_throughput_partitions_per_min = 5  # tunable
    executors_needed = math.ceil(partitions / (target_minutes * executor_throughput_partitions_per_min))
    print(f"  Executors needed: {executors_needed}")

    # step 3: map executors to nodes (r5.2xlarge: 2 executors each)
    executors_per_node = 2  # conservative (3 cores each, OS overhead)
    core_nodes = math.ceil(executors_needed / executors_per_node)
    print(f"  Core nodes (r5.2xlarge): {core_nodes}")

    # step 4: cost calculation
    inst = INSTANCE_TYPES['r5.2xlarge']
    master = INSTANCE_TYPES['m5.xlarge']
    duration_hr = target_minutes / 60

    # option A: all on-demand
    cost_od = (1 * master.on_demand_hr + core_nodes * inst.on_demand_hr) * duration_hr

    # option B: master on-demand + core/task nodes on spot (S3-based — safe)
    cost_spot = (1 * master.on_demand_hr + core_nodes * inst.spot_hr) * duration_hr

    print(f"  Duration: {target_minutes} min ({duration_hr:.2f}hr)")
    print(f"  Cost on-demand: ${cost_od:.2f}")
    print(f"  Cost with spot: ${cost_spot:.2f}  (saves {(1-cost_spot/cost_od)*100:.0f}%)")
    print(f"  spark.sql.shuffle.partitions = {partitions}")
    print(f"  spark.executor.memory = 24g  spark.executor.cores = 3")
    return core_nodes, cost_spot

print("=== SCENARIO 1: 500GB job, target 30 min ===")
size_emr_cluster(500, 30)

print()
print("=== SCENARIO 2: 2TB job, target 60 min ===")
size_emr_cluster(2000, 60)

print()
print("=== Spot vs On-Demand Risk Assessment ===")
roles = [
    ('Master node',  'on-demand', 'REQUIRED — interruption kills the cluster'),
    ('Core nodes',   'on-demand or spot (if S3 storage)', 'HDFS data lost if interrupted — use S3'),
    ('Task nodes',   'spot',      'Spark re-runs lost tasks — 0 risk with retries=2'),
]
for role, recommendation, reason in roles:
    print(f"  {role:20s}: {recommendation:35s} | {reason}")

print()
print("=== EMR vs Glue: when to use each ===")
print("  EMR:  custom Spark jars, specific Spark versions, >100 DPUs, spot instances")
print("  Glue: serverless, no infra mgmt, < 100 DPUs, pay-per-use, quick scripts")

print("\nEMR Spark sizing pattern complete.")

<a id='10'></a>
## 10. The Cloud Platforms Decision Map

---

```
USE CASE                           SERVICE        KEY CONFIG
────────────────────────────────────────────────────────────────────────────
Store raw data cheaply             S3 Standard    lifecycle → Glacier after 90d
SQL on data lake (ad-hoc)          Athena         partition + Parquet → 1000× cheaper
SQL on data lake (high freq)       Redshift Spec. uses Glue catalog, pays per scan
Dedicated BI / high-freq queries   Redshift        distkey = join key, sortkey = date
Serverless ETL                     Glue Jobs       10 DPU default, auto-scale
Heavy batch Spark                  EMR            spot task nodes, S3 storage
Schema registry                    Glue Catalog   central, Athena/RS Spectrum reads
Real-time ingestion                Kinesis+Firehose buffer → S3 every 5 min
Database migration                 DMS            CDC replication to Redshift/S3
────────────────────────────────────────────────────────────────────────────

COST OPTIMIZATION PRIORITY:
  1. Partition S3 data by query columns (biggest ROI)
  2. Convert CSV → Parquet + Snappy (10× storage + 80% Athena cost)
  3. EMR spot instances for task nodes (60-80% compute savings)
  4. Redshift reserved instances for always-on workloads (30% savings)
  5. S3 Lifecycle: Standard → IA (30d) → Glacier (90d)

REDSHIFT TUNING CHECKLIST:
  distkey = most common join column (or largest table join key)
  sortkey = most common WHERE/ORDER BY column
  ANALYZE COMPRESSION → set optimal column encoding (AZ64 default)
  VACUUM SORT ONLY after bulk load → restore sort order
  WLM queues: separate short queries from long-running ones
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for each service:

| Signal | Service |
|--------|----------|
| "Store data lake" | S3 + partition + Parquet |
| "Ad-hoc SQL, pay per query" | Athena |
| "Always-on BI / 1000s queries/day" | Redshift |
| "Serverless ETL" | Glue Job |
| "Heavy Spark, custom libs" | EMR |
| "Schema discovery" | Glue Crawler → Catalog |
| "Stream → S3 lake" | Kinesis Firehose |

---

### Key numbers — memorize these:

```
S3 Standard:          $0.023/GB/month
S3 Glacier Instant:   $0.004/GB/month  (80% cheaper)
Athena:               $5.00/TB scanned (10MB minimum)
Redshift dc2.large:   $0.25/node-hour
Glue:                 $0.44/DPU-hour   (min 10 min)
EMR spot savings:     60-80% vs on-demand
Parquet vs CSV:       10× smaller, 80-90% Athena cost reduction
Partition benefit:    90%+ scan reduction for date-range queries
```

---

### Common templates:

```python
# TEMPLATE: S3 partition path
s3_path = f's3://bucket/events/year={year}/month={month:02d}/day={day:02d}/'

# TEMPLATE: Redshift COPY from S3
# COPY table FROM 's3://bucket/path/' IAM_ROLE 'arn:...' FORMAT AS PARQUET;

# TEMPLATE: EMR cluster config key settings
spark_conf = {
    'spark.executor.memory': '24g',
    'spark.executor.cores': '3',
    'spark.sql.shuffle.partitions': '3900',  # data_gb * 1024 / 128
    'spark.dynamicAllocation.enabled': 'true',
}

# TEMPLATE: Athena workgroup cost cap
# CREATE WORKGROUP analytics WITH (
#   bytes_scanned_cutoff_per_query = 10737418240  -- 10GB
# );
```

---

### Gotchas to not forget:

```
❌  Athena on unpartitioned CSV — could cost hundreds per query on TB tables
❌  Redshift DISTKEY=ALL on large tables — duplicates entire table to every node
❌  EMR master node on spot — interrupted master = lost cluster
❌  Small S3 files (< 128MB) — use OPTIMIZE/compaction before Athena queries
✅  S3 as EMR storage → core nodes can be spot (no HDFS dependency)
✅  Glue bookmark tracks processed files — prevents reprocessing on restart
✅  Redshift Spectrum = query S3 from Redshift using Glue catalog — best of both
✅  VACUUM + ANALYZE after bulk loads → restores sort + updates stats
```


<a id='12'></a>
## 12. Summary Map

---

```
                   ☁️ AWS CLOUD DATA PLATFORMS
                              │
       ┌──────────────────────┼──────────────────────┐
       │                      │                      │
  STORAGE (S3)         CATALOG (Glue)          QUERY
  (Pattern 1)          (Pattern 2)        ┌────┴────┐
       │                    │          Athena    Redshift
  Partition strategy   Crawlers       (Pattern 4) (Pattern 3)
  year/month/day       ETL Jobs       Pay/scan   Always-on
  Lifecycle rules      Schema reg     Parquet    Distkey+sort
  Standard→Glacier     Hive compat.   Partition  Zone maps
                                         │
                                    COMPUTE (EMR)
                                    (Pattern 5)
                                         │
                                  Spark cluster
                                  Spot task nodes
                                  S3 storage → all spot
                                  Sizing formula

AWS DATA FLOW:
  Sources → Kinesis/DMS → S3 (raw)
          → Glue ETL    → S3 (curated/Parquet)
          → Athena (ad-hoc) / Redshift (frequent BI) ← Glue Catalog
          → SageMaker / Quicksight (serve)
```

---
*End of Cloud Data Platforms Master Guide — Sean Edition*
